In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tqdm
import sys
import os
import pandas as pd
import time

In [3]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("clip-ViT-B-32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /home/bkantz/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [5]:
sys.path.append("..")

In [6]:
from utils.datasets import DBPedia
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-09-02 12:40:11,239 - INFO - Loading faiss with AVX512 support.
2026-09-02 12:40:11,239 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-09-02 12:40:11,239 - INFO - Loading faiss with AVX2 support.
2026-09-02 12:40:11,239 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-09-02 12:40:11,240 - INFO - Loading faiss.
2026-09-02 12:40:11,251 - INFO - Successfully loaded faiss.


In [7]:
dataset = DBPedia(base_dir=Path("../data/dbpedia"))

In [8]:
test_label = "horse on the bow"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
test_tensor.to_literal().n3()

'"{\\"data\\": [-0.04262261837720871, -0.13338792324066162, 0.08612485975027084, -0.02091819792985916, -0.15041571855545044, 0.1737712323665619, -0.34851735830307007, -0.915214478969574, -0.3341490924358368, 0.17787931859493256, 0.05680549144744873, -0.2072562426328659, 0.5095283389091492, 0.18332259356975555, 0.11725617945194244, 0.11684612929821014, 0.2050066441297531, -0.147830069065094, -0.007135719060897827, 0.4961419701576233, 0.3664358854293823, 0.4141063094139099, -0.041074950248003006, 0.5009220838546753, -0.2138802707195282, -0.23931394517421722, -0.7052134275436401, 0.4793875813484192, -0.17963369190692902, 0.279243141412735, -0.06826923787593842, 0.08035053312778473, -0.20019766688346863, 0.25140517950057983, -0.16235414147377014, 0.11760087311267853, 0.10282528400421143, 0.20387886464595795, -0.04952246695756912, 0.26974374055862427, 0.1395796537399292, 0.532733678817749, 0.12486031651496887, -0.0857280045747757, 0.10658238083124161, -0.06904299557209015, 0.192440703511238

In [ ]:
db_qlever = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="QLever",
    port_offset=-100,
)
db_no_tensor_idx = QleverDBNative(
    id="qlever",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    enable_tensor_index=False,
    name="QLever (No Tensor Vocabulary)",
    port_offset=-200
)
db_fuseki = FusekiDBNative(
    id="fuseki",
    base_dir=Path(f"./scratch/{dataset.base_dir.name}"),
    dataset=dataset,
    use_encoded_ttl=True,
    name="Fuseki",
    exec_dir="../../jena-datatensor",
    port_offset=-300
)
possible_queries = db_qlever.get_queries(test_tensor)
print(possible_queries)
dbs: list[QleverDBNative | FusekiDBNative] = [db_qlever, db_no_tensor_idx, db_fuseki]
indices = ["index-encoded", "index-encoded-no-tidx", "fuseki-encoded"]
ids = ["dbpedia-encoded-tidx", "dbpedia-encoded", None]
for db, index, id in zip(dbs, indices, ids):
    db.db_dir = Path("../data/dbpedia/dbpedia") / index
    db.id = id if id is not None else db.id

TypeError: QleverDBNative.__init__() got an unexpected keyword argument 'timeout'

In [13]:
triple_counts = []
for db in dbs[:-1]:
    with db:
        count = db.get_triple_count()
        count_per_cls = db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?c (COUNT(?v) AS ?count) WHERE {
    ?s a ?c;
        dbo:thumbnail_embedding ?v .
    FILTER(STRSTARTS(STR(?c), "http://dbpedia.org/ontology/")) .
} GROUP BY ?c ORDER BY DESC(?count) LIMIT 100""")
        print(count_per_cls)
        count_tensors = db.query("""
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(?v) AS ?count) WHERE {
    ?s dbo:thumbnail_embedding ?v .
}""")["count"].values[0]
        print(f"{db.name}: {count} triples, {count_tensors} tensors")
        triple_counts.append({
            "count": count,
            "tensors": count_tensors,
            "db": db.name
        })
print(f"Total triples in DB: {sum([tc['count'] for tc in triple_counts])}")

2026-09-02 12:40:32,368 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded, base_dir=scratch/dbpedia
2026-09-02 12:40:32,369 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-09-02 12:40:32,369 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 12:40:32,369 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded already exists!
2026-09-02 12:40:32,369 - INFO - Stopping server!
2026-09-02 12:40:32,374 - ERROR - Command failed with return code 1
2026-09-02 12:40:32,374 - INFO - Starting QLever server on port 25946
2026-09-02 12:40:32,374 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-09-02 12:40:32,380 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)


2026-09-02 12:40:33,381 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 12:40:34,388 - INFO - Server is up and responding to queries
2026-09-02 12:40:39,453 - INFO - Stopping server!
25946/tcp:          
2026-09-02 12:40:39,474 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded-no-tidx, base_dir=scratch/dbpedia
2026-09-02 12:40:39,475 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-no-tidx/qlever-no-tidx_run.log
2026-09-02 12:40:39,475 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 12:40:39,475 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded-no-tidx already exists!
2026-09-02 12:40:39,476 - INFO - Stopping server!
2026-09-02 12:40:39,480 - ERROR - Command failed with return code 1
2026-09-02 12:40:39,480 - INFO - Starting QLever server on port 25847
2026-09-02 12:40:39,481 -

                      c   count
0             dbo:Place  435458
1          dbo:Location  435458
2           dbo:Species  350010
3         dbo:Eukaryote  349864
4            dbo:Animal  347640
..                  ...     ...
95    dbo:SoccerManager    4422
96   dbo:TelevisionShow    4389
97  dbo:MotorsportRacer    4336
98    dbo:SportsManager    4326
99       dbo:Locomotive    4318

[100 rows x 2 columns]
QLever: 421850341 triples, 2369261 tensors
 1847090

2026-09-02 12:40:40,482 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25847/qlever-no-tidx/sparql)
2026-09-02 12:40:41,487 - INFO - Server is up and responding to queries
2026-09-02 12:41:03,045 - INFO - Stopping server!


                      c   count
0             dbo:Place  435458
1          dbo:Location  435458
2           dbo:Species  350010
3         dbo:Eukaryote  349864
4            dbo:Animal  347640
..                  ...     ...
95    dbo:SoccerManager    4422
96   dbo:TelevisionShow    4389
97  dbo:MotorsportRacer    4336
98    dbo:SportsManager    4326
99       dbo:Locomotive    4318

[100 rows x 2 columns]
QLever (No Tensor Vocabulary): 421850341 triples, 2369261 tensors
Total triples in DB: 843700682


25847/tcp:          


 1847265

In [14]:
count_per_cls.to_csv("../data/dbpedia/thumbnail_embedding_count_per_cls.csv", index=False)

In [15]:
counts_df = pd.DataFrame(triple_counts)
counts_df

,count,tensors,db
0,421850341,2369261,QLever
1,421850341,2369261,QLever (No Tensor Vocabulary)


In [16]:
counts_df["power"] = counts_df["count"].apply(lambda x: int(np.log10(x)))
counts_df["size"] = counts_df["count"]
counts_df["full_size"] = counts_df["count"]
counts_df["full_number_of_tensors"] = counts_df["tensors"]

In [17]:
from utils.helpers import pretty_print_counts

out_path_counts = Path("../scratch/results") / "dbpedia_counts.tex"
pretty_print_counts(counts_df, out_path_counts)

Format cols: ['power', 'size', 'full_size', 'full_number_of_tensors'] Non-format cols: []


,Power,Generation $t$,$n$,$n_{tensors}$
0,8,421850341,421850341,2369261
1,8,421850341,421850341,2369261


In [45]:
with db_qlever as db:
    timings = []
    for _ in tqdm.tqdm(range(5)):
        start = time.time()
        r = db.query_auto(
            query_difficulty=QUERY_DIFFICULTY.HARD,
            query_type=QUERY_TYPE.INDEX,
            tensor=test_tensor,
            timeout=120
        )
        end = time.time()
        timings.append(end - start)
print("Timings avg:", sum(timings) / len(timings))
r

2026-09-02 13:14:47,700 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded, base_dir=scratch/dbpedia
2026-09-02 13:14:47,701 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-09-02 13:14:47,701 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 13:14:47,701 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded already exists!
2026-09-02 13:14:47,701 - INFO - Stopping server!
2026-09-02 13:14:47,706 - ERROR - Command failed with return code 1
2026-09-02 13:14:47,706 - INFO - Starting QLever server on port 25946
2026-09-02 13:14:47,706 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-09-02 13:14:47,707 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)


2026-09-02 13:14:48,708 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 13:14:49,713 - INFO - Server is up and responding to queries
100%|██████████| 5/5 [00:01<00:00,  3.84it/s]
2026-09-02 13:14:51,016 - INFO - Stopping server!


 1904229Timings avg: 0.2603337287902832


25946/tcp:          


,r,s,dist,thumb_rail_emb,thumb_ship_emb
0,dbr:Hathlangoo,dbr:Ngang_Pass,147.8597106934,"{""data"": [0.1213223785161972, 0.67646539211273...","{""data"": [0.266842782497406, 0.282170355319976..."
1,dbr:Playa_Mayabeque,dbr:Welland_Canal,147.4058380127,"{""data"": [0.09042239189147949, 0.0884934589266...","{""data"": [0.14147435128688812, -0.177494421601..."
2,dbr:Sihri,dbr:Speikboden_(South_Tyrol),139.6905670166,"{""data"": [-0.02784011885523796, -0.11094573140...","{""data"": [0.010983843356370926, -0.03950532525..."
3,"dbr:Big_Pine_Key,_Florida",dbr:Japanese_Alps,137.8491210938,"{""data"": [0.013072215020656586, 0.038753185421...","{""data"": [0.03383301943540573, -0.152067333459..."
4,dbr:Deobahal,dbr:Ngang_Pass,136.5958251953,"{""data"": [-0.13472652435302734, 0.181089758872...","{""data"": [0.266842782497406, 0.282170355319976..."
5,"dbr:Sherwood,_Michigan",dbr:Furtbek,136.1393127441,"{""data"": [-0.19389194250106812, -0.00306503474...","{""data"": [-0.2586419880390167, 0.2508565783500..."
6,"dbr:Hubbard_Lake,_Alcona_County,_Michigan",dbr:Welland_Canal,135.5609436035,"{""data"": [-0.1926412433385849, -0.158066034317...","{""data"": [0.14147435128688812, -0.177494421601..."
7,"dbr:Lost_Lake_Woods,_Michigan",dbr:Welland_Canal,135.1779022217,"{""data"": [-0.1862151026725769, -0.135209918022...","{""data"": [0.14147435128688812, -0.177494421601..."
8,"dbr:Natural_Bridge_Station,_Virginia",dbr:Ngang_Pass,134.256149292,"{""data"": [-0.024006932973861694, 0.29700988531...","{""data"": [0.266842782497406, 0.282170355319976..."
9,"dbr:Pepin,_Wisconsin",dbr:Welland_Canal,134.02003479,"{""data"": [0.26799505949020386, 0.0406789481639...","{""data"": [0.14147435128688812, -0.177494421601..."


In [ ]:
with db_qlever as db:
    timings = []
    for _ in tqdm.tqdm(range(5)):
        start = time.time()
        r = db.query_auto(
            query_difficulty=QUERY_DIFFICULTY.HARD,
            query_type=QUERY_TYPE.INDEX,
            tensor=test_tensor,
        )
        end = time.time()
        timings.append(end - start)
print("Timings avg:", sum(timings) / len(timings))
r

2026-05-18 12:42:30,669 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-05-18 12:42:30,669 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-05-18 12:42:30,670 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-05-18 12:42:30,671 - INFO - Stopping server!
2026-05-18 12:42:30,726 - ERROR - Command failed with return code 1
2026-05-18 12:42:30,727 - INFO - Starting QLever server on port 25943
2026-05-18 12:42:30,727 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-05-18 12:42:30,730 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:42:31,732 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)

Timings avg: 1.2492853164672852


,r,s,dist,thumb_rail_emb,thumb_ship_emb
0,dbr:Hathlangoo,dbr:Ngang_Pass,147.8597106934,"{""data"": [0.1213223785161972, 0.67646539211273...","{""data"": [0.266842782497406, 0.282170355319976..."
1,dbr:Playa_Mayabeque,dbr:Welland_Canal,147.4058380127,"{""data"": [0.09042239189147949, 0.0884934589266...","{""data"": [0.14147435128688812, -0.177494421601..."
2,dbr:Sihri,dbr:Speikboden_(South_Tyrol),139.6905670166,"{""data"": [-0.02784011885523796, -0.11094573140...","{""data"": [0.010983843356370926, -0.03950532525..."
3,"dbr:Big_Pine_Key,_Florida",dbr:Japanese_Alps,137.8491210938,"{""data"": [0.013072215020656586, 0.038753185421...","{""data"": [0.03383301943540573, -0.152067333459..."
4,dbr:Deobahal,dbr:Ngang_Pass,136.5958251953,"{""data"": [-0.13472652435302734, 0.181089758872...","{""data"": [0.266842782497406, 0.282170355319976..."
5,"dbr:Sherwood,_Michigan",dbr:Furtbek,136.1393127441,"{""data"": [-0.19389194250106812, -0.00306503474...","{""data"": [-0.2586419880390167, 0.2508565783500..."
6,"dbr:Hubbard_Lake,_Alcona_County,_Michigan",dbr:Welland_Canal,135.5609436035,"{""data"": [-0.1926412433385849, -0.158066034317...","{""data"": [0.14147435128688812, -0.177494421601..."
7,"dbr:Lost_Lake_Woods,_Michigan",dbr:Welland_Canal,135.1779022217,"{""data"": [-0.1862151026725769, -0.135209918022...","{""data"": [0.14147435128688812, -0.177494421601..."
8,"dbr:Natural_Bridge_Station,_Virginia",dbr:Ngang_Pass,134.256149292,"{""data"": [-0.024006932973861694, 0.29700988531...","{""data"": [0.266842782497406, 0.282170355319976..."
9,"dbr:Pepin,_Wisconsin",dbr:Welland_Canal,134.02003479,"{""data"": [0.26799505949020386, 0.0406789481639...","{""data"": [0.14147435128688812, -0.177494421601..."


In [29]:
with dbs[0] as db:
    res_native = db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
SELECT * WHERE {{
    ?s a dbo:{dataset.left} .
    ?s dbo:thumbnail_embedding ?thumb_emb .
    ?s dbo:thumbnail_original ?thumb .      
    BIND(dtf:dotProduct(?thumb_emb, {test_tensor.to_literal().n3()}) AS ?dist)
}} 
ORDER BY DESC(?dist)
LIMIT 10""")
res_native

2026-09-02 12:58:22,449 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded, base_dir=scratch/dbpedia
2026-09-02 12:58:22,450 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-09-02 12:58:22,450 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 12:58:22,450 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded already exists!
2026-09-02 12:58:22,450 - INFO - Stopping server!
2026-09-02 12:58:22,455 - ERROR - Command failed with return code 1
2026-09-02 12:58:22,455 - INFO - Starting QLever server on port 25946
2026-09-02 12:58:22,455 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 16G --tensor-search-max-num-threads 4


2026-09-02 12:58:22,456 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 12:58:23,456 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 12:58:24,462 - INFO - Server is up and responding to queries
2026-09-02 12:58:25,235 - INFO - Stopping server!


 1879986

25946/tcp:          


,s,thumb_emb,thumb,dist
0,dbr:Kundudo,"{""data"": [-0.20942936837673187, 0.028205197304...",http://upload.wikimedia.org/wikipedia/commons/...,28.98929405212
1,dbr:Kundudo,"{""data"": [-0.20942936837673187, 0.028205197304...",https://upload.wikimedia.org/wikipedia/commons...,28.98929405212
2,dbr:Kundudo,"{""data"": [-0.20942936837673187, 0.028205197304...",https://upload.wikimedia.org/wikipedia/commons...,28.98929405212
3,dbr:Ohio_and_Erie_Canal,"{""data"": [-0.17475301027297974, 0.018981928005...",https://upload.wikimedia.org/wikipedia/commons...,28.41627120972
4,dbr:Ohio_and_Erie_Canal,"{""data"": [-0.17475301027297974, 0.018981928005...",http://upload.wikimedia.org/wikipedia/commons/...,28.41627120972
5,dbr:Ohio_and_Erie_Canal,"{""data"": [-0.17475301027297974, 0.018981928005...",http://upload.wikimedia.org/wikipedia/commons/...,28.41627120972
6,dbr:Chesapeake_and_Ohio_Canal,"{""data"": [-0.42376554012298584, 0.671011090278...",https://upload.wikimedia.org/wikipedia/commons...,27.86582565308
7,dbr:Chesapeake_and_Ohio_Canal,"{""data"": [-0.42376554012298584, 0.671011090278...",https://upload.wikimedia.org/wikipedia/commons...,27.86582565308
8,dbr:Chesapeake_and_Ohio_Canal,"{""data"": [-0.42376554012298584, 0.671011090278...",http://upload.wikimedia.org/wikipedia/commons/...,27.86582565308
9,dbr:Chesapeake_and_Ohio_Canal,"{""data"": [-0.42376554012298584, 0.671011090278...",http://upload.wikimedia.org/wikipedia/commons/...,27.86582565308


In [30]:
with dbs[0] as db:
    res =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT * WHERE {{
SERVICE tensorIndex: {{
    _:config tensorIndex:numNN 10 ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?s, ?thumb ;
    tensorIndex:searchK 64 ;
    tensorIndex:kIVF 128 ;
    # tensorIndex:experimentalRightCacheName "easy_index_dbpedia" ;
    tensorIndex:right ?thumb_emb ;
    tensorIndex:algorithm tensorIndex:ivf ;
    tensorIndex:distance tensorIndex:dot .
       {{
            ?s a dbo:{dataset.left} ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail_original ?thumb .
        }}
    }}
    VALUES (?query_vector) {{ ({test_tensor.to_literal().n3()}) }}
}} 
ORDER BY DESC(?dist)
""")
res

2026-09-02 12:58:34,911 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded, base_dir=scratch/dbpedia
2026-09-02 12:58:34,912 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-09-02 12:58:34,912 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 12:58:34,912 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded already exists!
2026-09-02 12:58:34,912 - INFO - Stopping server!
2026-09-02 12:58:34,935 - ERROR - Command failed with return code 1
2026-09-02 12:58:34,936 - INFO - Starting QLever server on port 25946
2026-09-02 12:58:34,936 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-09-02 12:58:34,937 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)


2026-09-02 12:58:35,938 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 12:58:36,943 - INFO - Server is up and responding to queries
2026-09-02 12:58:37,580 - INFO - Stopping server!


 1880302

25946/tcp:          


,query_vector,dist,s,thumb,thumb_emb
0,"{""data"": [-0.04262261837720871, -0.13338792324...",28.98929023743,dbr:Kundudo,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.20942936837673187, 0.028205197304..."
1,"{""data"": [-0.04262261837720871, -0.13338792324...",28.98929023743,dbr:Kundudo,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.20942936837673187, 0.028205197304..."
2,"{""data"": [-0.04262261837720871, -0.13338792324...",28.98929023743,dbr:Kundudo,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.20942936837673187, 0.028205197304..."
3,"{""data"": [-0.04262261837720871, -0.13338792324...",28.41626548767,dbr:Ohio_and_Erie_Canal,https://upload.wikimedia.org/wikipedia/commons...,"{""data"": [-0.17475301027297974, 0.018981928005..."
4,"{""data"": [-0.04262261837720871, -0.13338792324...",28.41626548767,dbr:Ohio_and_Erie_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.17475301027297974, 0.018981928005..."
5,"{""data"": [-0.04262261837720871, -0.13338792324...",28.41626548767,dbr:Ohio_and_Erie_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.17475301027297974, 0.018981928005..."
6,"{""data"": [-0.04262261837720871, -0.13338792324...",27.86582565308,dbr:Chesapeake_and_Ohio_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.42376554012298584, 0.671011090278..."
7,"{""data"": [-0.04262261837720871, -0.13338792324...",27.86582565308,dbr:Chesapeake_and_Ohio_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.42376554012298584, 0.671011090278..."
8,"{""data"": [-0.04262261837720871, -0.13338792324...",27.86582565308,dbr:Chesapeake_and_Ohio_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.42376554012298584, 0.671011090278..."
9,"{""data"": [-0.04262261837720871, -0.13338792324...",27.86582565308,dbr:Chesapeake_and_Ohio_Canal,http://upload.wikimedia.org/wikipedia/commons/...,"{""data"": [-0.42376554012298584, 0.671011090278..."


In [41]:
res.to_csv(Path("scratch") / "dbpedia_results_index.csv", index=False)

In [22]:
with dbs[1] as db:
    db.query_auto(test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.HARD)

2026-05-18 12:42:59,438 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-no-tidx/qlever-no-tidx_run.log
2026-05-18 12:42:59,439 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-05-18 12:42:59,440 - WARNING - DB directory ../data/dbpedia/index-encoded-no-tidx already exists!
2026-05-18 12:42:59,441 - INFO - Stopping server!
2026-05-18 12:42:59,454 - ERROR - Command failed with return code 1
2026-05-18 12:42:59,455 - INFO - Starting QLever server on port 25844
2026-05-18 12:42:59,455 - INFO - Running command: qlever-server -i dbpedia-encoded --port 25844 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-05-18 12:42:59,458 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026-05-18 12:43:00,459 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25844/qlever-no-tidx/sparql)
2026

In [23]:
from utils.helpers import ndcgscore_query
reference_result = res_native
score = ndcgscore_query(res, reference_result, k=10)
print(f"Score: {score}")

Score: 0.8042472959280005


In [24]:
# count ships


with dbs[0] as db:
    rs =db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:{dataset.left} ;
            dbo:thumbnail_embedding ?thumb_emb .
}} LIMIT 10""")
    rr =  db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT (COUNT(*) AS ?count) WHERE {{ 
            ?s a dbo:{dataset.right} ;
            dbo:thumbnail_embedding ?thumb_emb .
}} LIMIT 10""")
rs.iloc[0], rr.iloc[0]

2026-05-18 12:43:15,939 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-05-18 12:43:15,943 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-05-18 12:43:15,943 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-05-18 12:43:15,946 - INFO - Stopping server!


2026-05-18 12:43:16,053 - ERROR - Command failed with return code 1
2026-05-18 12:43:16,054 - INFO - Starting QLever server on port 25943
2026-05-18 12:43:16,056 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-05-18 12:43:16,058 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:43:17,059 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:43:18,061 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:43:19,063 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:43:20,065 - I

(count    46351
 Name: 0, dtype: str,
 count    31149
 Name: 0, dtype: str)

In [40]:
# for find similar thumbnails between ships and railroads
times = []
with dbs[0] as db:
    #     warmup = db.query("""
    # PREFIX dbr: <http://dbpedia.org/resource/>
    # PREFIX dbo: <http://dbpedia.org/ontology/>
    # PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    # SELECT * WHERE {
    #     { SELECT * WHERE {
    #         ?r a dbo:Work .
    #         ?r dbo:thumbnail_embedding ?thumb_rail_emb .
    #         ?r dbo:thumbnail_original ?thumb_rail .
    #     } LIMIT 1
    #     }
    #     SERVICE tensorIndex: {
    #     _:config tensorIndex:numNN 10 ;
    #     tensorIndex:left ?thumb_rail_emb ;
    #     tensorIndex:bindDistance ?dist ;
    #     tensorIndex:payload ?s, ?thumb_ship ;
    #     tensorIndex:searchK 1 ;
    #     tensorIndex:nTrees 128 ;
    #     tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
    #     tensorIndex:right ?thumb_emb_ship ;
    #     tensorIndex:algorithm tensorIndex:ivf ;
    #     tensorIndex:distance tensorIndex:dot .
    #         {
    #             ?s a dbo:Ship ;
    #             dbo:thumbnail_embedding ?thumb_emb_ship ;
    #             dbo:thumbnail_original ?thumb_ship .
    #         }
    #     }
    #     }""")
    for _ in range(5):
        start = time.time()

        noised_tensor = DataTensor.from_numpy(
            test_tensor.data + np.random.normal(scale=0.001, size=test_tensor.shape)
        )
        railway_ship_assoc = db.query(f"""
    PREFIX dbr: <http://dbpedia.org/resource/>
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
    PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
    SELECT DISTINCT ?r ?s  ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
        {{
        SELECT DISTINCT ?r ?s ?dist ?thumb_rail_emb ?thumb_ship_emb WHERE {{
            
            ?r a dbo:{dataset.left} ;
             dbo:thumbnail_embedding ?thumb_rail_emb .
                                        
            SERVICE tensorIndex: {{
            _:config tensorIndex:numNN 1 ;
            tensorIndex:left ?thumb_rail_emb ;
            tensorIndex:bindDistance ?dist ;
            tensorIndex:payload ?s, ?thumb_ship ;
            tensorIndex:searchK 1 ;
            tensorIndex:kIVF 128 ;
            tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
            tensorIndex:right ?thumb_ship_emb ;
            tensorIndex:algorithm tensorIndex:ivf ;
            tensorIndex:distance tensorIndex:dot .
            {{
                ?s a dbo:{dataset.right} ;
                dbo:thumbnail_embedding ?thumb_ship_emb ;    
            }}
            }}                  
        }}
        }}
        VALUES (?some_emb) {{ ({noised_tensor.to_literal().n3()}) }}
                                    
    }}
    ORDER BY DESC(?dist)
    LIMIT 20""")
        end = time.time()
        delta = end - start
        times.append(delta)
times

2026-09-02 13:06:59,100 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/dbpedia/index-encoded, base_dir=scratch/dbpedia
2026-09-02 13:06:59,101 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-09-02 13:06:59,101 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-09-02 13:06:59,101 - WARNING - DB directory ../data/dbpedia/dbpedia/index-encoded already exists!
2026-09-02 13:06:59,102 - INFO - Stopping server!
2026-09-02 13:06:59,107 - ERROR - Command failed with return code 1
2026-09-02 13:06:59,107 - INFO - Starting QLever server on port 25946
2026-09-02 13:06:59,107 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25946 -k 0 -m 16G --tensor-search-max-num-threads 4


2026-09-02 13:06:59,108 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 13:07:00,109 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25946/qlever-with-tidx/sparql)
2026-09-02 13:07:01,114 - INFO - Server is up and responding to queries
2026-09-02 13:07:31,125 - INFO - Stopping server!


 1893142

25946/tcp:          


TimeoutError: 

In [34]:
railway_ship_assoc

NameError: name 'railway_ship_assoc' is not defined

In [26]:
with db_qlever as db:
    hard_results = db.query(
        f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?l ?r ?dist ?thumb_left ?thumb_right ?thumb_left_emb ?thumb_right_emb WHERE {{
    {{
    SELECT DISTINCT ?l ?r ?dist ?thumb_left ?thumb_right ?thumb_left_emb ?thumb_right_emb WHERE {{

        ?l a dbo:{dataset.left} .
        ?l dbo:thumbnail_embedding ?thumb_left_emb .
        ?l dbo:thumbnail ?thumb_left .

        SERVICE tensorIndex: {{
        _:config tensorIndex:numNN 1 ;
        tensorIndex:left ?thumb_left_emb ;
        tensorIndex:bindDistance ?dist ;
        tensorIndex:payload ?r, ?thumb_right_emb, ?thumb_right ;
        tensorIndex:searchK 1 ;
        tensorIndex:kIVF 512 ;
        tensorIndex:experimentalRightCacheName "hard_index_dbpedia" ;
        tensorIndex:right ?thumb_right_emb ;
        tensorIndex:algorithm tensorIndex:ivf ;
        tensorIndex:distance tensorIndex:dot .
        {{
            ?r a dbo:{dataset.right} ;
            dbo:thumbnail_embedding ?thumb_right_emb ;
            dbo:thumbnail ?thumb_right .
        }}
        }}
    }}
    }}
}}
ORDER BY DESC(?dist)
LIMIT 5"""
    )
    #     # hard_results = db.query_auto(
    #     #     test_tensor, query_type=QUERY_TYPE.INDEX, query_difficulty=QUERY_DIFFICULTY.HARD
    #     # )
    easy_results = db.query(f"""
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?l ?thumb_emb ?dist ?thumb WHERE {{
SERVICE tensorIndex: {{
    _:config tensorIndex:numNN 10 ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?l, ?thumb_emb, ?thumb ;
    tensorIndex:searchK 1 ;
    tensorIndex:experimentalRightCacheName "easy_index_dbpedia" ;
    tensorIndex:right ?thumb_emb ;
    tensorIndex:algorithm tensorIndex:ivf ;
    tensorIndex:distance tensorIndex:dot .
       {{
            ?l a dbo:{dataset.left} ;
            dbo:thumbnail_embedding ?thumb_emb ;
            dbo:thumbnail ?thumb .
        }}
    }}
VALUES (?query_vector) {{ ({test_tensor.to_literal().n3()}) }}
}} 
ORDER BY DESC(?dist)
                            """)

2026-05-18 12:43:28,160 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-05-18 12:43:28,161 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-05-18 12:43:28,162 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-05-18 12:43:28,162 - INFO - Stopping server!
2026-05-18 12:43:28,211 - ERROR - Command failed with return code 1
2026-05-18 12:43:28,212 - INFO - Starting QLever server on port 25943
2026-05-18 12:43:28,212 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-05-18 12:43:28,214 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-05-18 12:43:29,217 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)

In [27]:
hard_results.to_csv(Path("scratch") / "dbpedia_hard_results.csv", index=False)
hard_results

,l,r,dist,thumb_left,thumb_right,thumb_left_emb,thumb_right_emb
0,dbr:Ngang_Pass,dbr:Hathlangoo,147.8597106934,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,"{""data"": [0.266842782497406, 0.282170355319976...","{""data"": [0.1213223785161972, 0.67646539211273..."
1,dbr:Welland_Canal,dbr:Playa_Mayabeque,147.4058380127,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,"{""data"": [0.14147435128688812, -0.177494421601...","{""data"": [0.09042239189147949, 0.0884934589266..."
2,dbr:Speikboden_(South_Tyrol),dbr:Playa_Mayabeque,143.0099334717,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,"{""data"": [0.010983843356370926, -0.03950532525...","{""data"": [0.09042239189147949, 0.0884934589266..."
3,dbr:Rim_(crater),dbr:Playa_Mayabeque,140.2051239014,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,"{""data"": [-0.09978093951940536, 0.058597564697...","{""data"": [0.09042239189147949, 0.0884934589266..."
4,dbr:Wittstrauch,dbr:Hathlangoo,139.259979248,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,"{""data"": [0.19549092650413513, 0.1191306039690...","{""data"": [0.1213223785161972, 0.67646539211273..."


In [28]:
easy_results.to_csv(Path("scratch") / "dbpedia_easy_results.csv", index=False)
easy_results

,l,thumb_emb,dist,thumb
0,dbr:Swin_(Thames),"{""data"": [0.3809254765510559, -0.0876748263835...",27.77111434937,http://commons.wikimedia.org/wiki/Special:File...
1,dbr:Lake_Balaton,"{""data"": [0.18073731660842896, -0.051469188183...",27.32340812683,http://commons.wikimedia.org/wiki/Special:File...
2,dbr:Lake_Balaton,"{""data"": [0.3280792236328125, 0.13119816780090...",26.78946304321,http://commons.wikimedia.org/wiki/Special:File...
3,dbr:North_Sea,"{""data"": [0.22238808870315552, -0.094367086887...",26.64107513428,http://commons.wikimedia.org/wiki/Special:File...
4,dbr:Lennox_Passage_(waterway),"{""data"": [0.23200394213199615, -0.242229431867...",26.32609176636,http://commons.wikimedia.org/wiki/Special:File...
5,dbr:Gulf_of_Thailand,"{""data"": [0.510789692401886, -0.14017504453659...",26.1806640625,http://commons.wikimedia.org/wiki/Special:File...
6,dbr:Sproat_Lake,"{""data"": [0.5528449416160583, 0.43867826461791...",26.17969512939,http://commons.wikimedia.org/wiki/Special:File...
7,dbr:Pulicat_Lake,"{""data"": [0.6338863372802734, 0.07557176053524...",26.14479255676,http://commons.wikimedia.org/wiki/Special:File...
8,dbr:Chao_Phraya_River,"{""data"": [0.5662512183189392, 0.35592082142829...",25.5197429657,http://commons.wikimedia.org/wiki/Special:File...
9,dbr:Gulf_of_Mexico,"{""data"": [0.28293147683143616, -0.315052270889...",25.28795051575,http://commons.wikimedia.org/wiki/Special:File...


In [29]:
easy_results.loc[0, "s"], easy_results.loc[0, "thumb_emb"], easy_results.loc[0, "dist"]

KeyError: 's'

In [ ]:
hard_results

,r,s,dist,thumb_work_emb,thumb_settlement_emb
0,dbr:Chengdu_Metro,dbr:Virginia-class_submarine,149.7560424805,"{""data"": [0.22622768580913544, 0.0546032562851...","{""data"": [-0.1000426858663559, -0.080208711326..."
1,dbr:Taipa_line,dbr:Virginia-class_submarine,148.5401763916,"{""data"": [0.03102342039346695, -0.059559382498...","{""data"": [-0.1000426858663559, -0.080208711326..."
2,dbr:Yangluo_Line,dbr:Virginia-class_submarine,146.5701904297,"{""data"": [0.13836224377155304, 0.1058328151702...","{""data"": [-0.1000426858663559, -0.080208711326..."
3,dbr:Buenos_Aires_Underground,dbr:Virginia-class_submarine,140.4429473877,"{""data"": [-0.057316623628139496, -0.1589123308...","{""data"": [-0.1000426858663559, -0.080208711326..."
4,dbr:Line_13_(CPTM),dbr:Virginia-class_submarine,139.5857849121,"{""data"": [-0.21847796440124512, -0.04886087775...","{""data"": [-0.1000426858663559, -0.080208711326..."
5,dbr:Orlyval,dbr:Virginia-class_submarine,138.4967041016,"{""data"": [0.11910116672515869, -0.394591093063...","{""data"": [-0.1000426858663559, -0.080208711326..."
6,dbr:Abbey_Line,dbr:Virginia-class_submarine,137.7716217041,"{""data"": [0.17977674305438995, -0.300230383872...","{""data"": [-0.1000426858663559, -0.080208711326..."
7,dbr:Kolkata_Metro,dbr:Virginia-class_submarine,137.671875,"{""data"": [-0.012433663010597229, -0.1900815814...","{""data"": [-0.1000426858663559, -0.080208711326..."
8,dbr:London_Overground,dbr:Virginia-class_submarine,137.6464233398,"{""data"": [0.02959146350622177, 0.0380432158708...","{""data"": [-0.1000426858663559, -0.080208711326..."
9,dbr:CDGVAL,dbr:Virginia-class_submarine,135.4984130859,"{""data"": [-0.17296558618545532, 0.003928374499...","{""data"": [-0.1000426858663559, -0.080208711326..."


In [ ]:
# load images for both results
from utils.dbs.base_db import BaseDB


def enhance_col_with_thumbs(
    db: BaseDB, df: pd.DataFrame, dbr_col: str, thumb_col_name="thumb"
) -> pd.DataFrame:

    thumbs = []
    g = tqdm.tqdm(df[dbr_col], desc=f"Fetching thumbnails for {thumb_col_name}")
    for s in g:
        g.set_description(f"Fetching thumbnails for {thumb_col_name}: '{s}'")
        s = f"<{s.replace('dbr:', 'http://dbpedia.org/resource/')}>"
        res = db.query(f"""PREFIX dbr: <http://dbpedia.org/resource/>
                            PREFIX dbo: <http://dbpedia.org/ontology/>
                            SELECT ?thumb ?thumb_lc WHERE {{
                                {s} dbo:thumbnail_original ?thumb .
                                BIND (LCASE(STR(?thumb)) AS ?thumb_lc) 
                                FILTER (STRENDS(?thumb_lc, ".png") || STRENDS(?thumb_lc, ".jpg") || STRENDS(?thumb_lc, ".jpeg"))
                            }} LIMIT 1""")
        # print(f"Query result for {s}: {res}")
        if len(res) > 0:
            thumbs.append(res["thumb"].values[0])
        else:
            thumbs.append(None)
    df[thumb_col_name] = thumbs
    return df


with db_qlever as db:
    easy_results = enhance_col_with_thumbs(db, easy_results, "s")
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "s", thumb_col_name="thumb_s"
    )
    hard_results = enhance_col_with_thumbs(
        db, hard_results, "r", thumb_col_name="thumb_r"
    )



2026-04-27 15:54:17,241 - INFO - Logging QLever setup to scratch/dbpedia/db/qlever-with-tidx/qlever-with-tidx_run.log
2026-04-27 15:54:17,242 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt.gz
2026-04-27 15:54:17,243 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-04-27 15:54:17,244 - INFO - Stopping server!
2026-04-27 15:54:17,314 - ERROR - Command failed with return code 1
2026-04-27 15:54:17,315 - INFO - Starting QLever server on port 25943
2026-04-27 15:54:17,315 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 25943 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-27 15:54:17,318 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/sparql)
2026-04-27 15:54:18,319 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:25943/qlever-with-tidx/spar

In [ ]:
easy_results.loc[0, 'thumb'].split('/')[-1]

'HMS_Warrior_Pembroke_Dock_July_1977_B.jpg'

## Token

You need to set your Oauth tokens in the local `.env`:
```sh
MEDIA_WIKI_TOKEN="..."
MEDIA_WIKI_SECRET="..."
MEDIA_WIKI_ACCESS_TOKEN="..."
MEDIA_WIKI_ACCESS_SECRET="..."
```
You can register a new token [here](https://meta.wikimedia.org/wiki/Special:OAuthConsumerRegistration/propose/oauth1a).

In [ ]:
import requests
from requests_oauthlib import OAuth1
import dotenv
from utils.datasets.dbpedia_utils.helpers import USER_AGENT

dotenv.load_dotenv()  # to load MEDIAWIKI_TOKEN from .env file

auth = OAuth1(
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)   

In [ ]:
auth.client

<Client client_key=eea9f51d9008d744c272c3ece851b184, client_secret=****, resource_owner_key=914b284deef1f1f0525ea803d55914b0, resource_owner_secret=****, signature_method=HMAC-SHA1, signature_type=AUTH_HEADER, callback_uri=None, rsa_key=None, verifier=None, realm=None, encoding=utf-8, decoding=utf-8, nonce=None, timestamp=None>

In [ ]:
url=f"https://commons.wikimedia.org/wiki/Special:FilePath/{easy_results.loc[0, 'thumb'].split('/')[-1]}?width=330"
resp = requests.get(
    url,
    auth=auth,
    headers={"User-Agent": USER_AGENT},
)

In [ ]:
import pywikibot

pywikibot.config.usernames['commons']['commons'] = "Dakantz"
# set user-agent

authenticate = (
    os.getenv("MEDIA_WIKI_TOKEN"),
    os.getenv("MEDIA_WIKI_SECRET"),
    os.getenv("MEDIA_WIKI_ACCESS_TOKEN"),
    os.getenv("MEDIA_WIKI_ACCESS_SECRET"),
)
pywikibot.config.authenticate['commons.wikimedia.org'] = authenticate
pywikibot.config.user_agent = USER_AGENT
site = pywikibot.Site('commons', 'commons')
site.login()

In [ ]:
site.allimages()

In [ ]:
from PIL import Image
fname = easy_results.loc[0, "thumb"].split("/")[-1]
img = pywikibot.FilePage(site, fname)

img.download(filename=fname, url_width=330)
img = Image.open(fname)

In [ ]:
from utils.datasets.dbpedia_utils import image_formatter


def thumbs_to_pil(
    thumbs: pd.Series, scratch_dir=Path("../scratch/dbpedia_thumbs")
) -> pd.Series:

    scratch_dir.mkdir(parents=True, exist_ok=True)
    pil_images = []
    g = tqdm.tqdm(thumbs, desc="Fetching thumbnails")
    for thumb in g:
        g.set_description(f"Fetching thumbnail for {thumb}...")
        # url_thumb = get_wc_thumb(thumb)

        fname = thumb.split("/")[-1]
        out_f = scratch_dir / fname
        if not out_f.exists():
            img = pywikibot.FilePage(site, fname)
            img.download(filename=out_f, url_width=330)
        img = Image.open(out_f)
        pil_images.append(img)
    return pd.Series(pil_images, index=thumbs.index)


easy_results["thumb_img"] = thumbs_to_pil(easy_results["thumb"])
hard_results["thumb_s_img"] = thumbs_to_pil(hard_results["thumb_s"])
hard_results["thumb_r_img"] = thumbs_to_pil(hard_results["thumb_r"])

Fetching thumbnail for https://upload.wikimedia.org/wikipedia/commons/2/23/HMS_Warrior_Pembroke_Dock_July_1977_B.jpg...:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/0/0d/SS_Great_Britain_diagram.jpg...:  40%|████      | 4/10 [00:08<00:12,  2.15s/it]              WARNING: Http response status 429
Fetching thumbnail for http://upload.wikimedia.org/wikipedia/commons/0/0d/SS_Great_Britain_diagram.jpg...:  40%|████      | 4/10 [00:11<00:16,  2.79s/it]


FileNotFoundError: [Errno 2] No such file or directory: '../scratch/dbpedia_thumbs/SS_Great_Britain_diagram.jpg'

In [ ]:
out_dir = Path("../scratch/results/")


def to_tex_with_thumbs(
    df,
    t_cols=["thumb_img"],
    col_mapping={
        "s": "Ship",
        "r": "Railway",
        "thumb_img_tex": "Thumbnail",
        "dist": "Distance",
    },
    base_dir="figures/generated",
    sub_dir="dbpedia/media",
    out_name="dbpedia_easy_results.tex",
    col_format ="lp{3cm}r"
):
    out_dir.mkdir(parents=True, exist_ok=True)
    results_dir = Path(out_dir) / sub_dir
    results_dir.mkdir(parents=True, exist_ok=True)
    fig_dir = Path(base_dir) / sub_dir
    for t_col in t_cols:
        df[f"{t_col}_tex"] = df[t_col]
        
        for i, r in df.iterrows():
            if r[t_col] is not None and isinstance(r[t_col], Image.Image):
                fname = f"{t_col}_{i}.png"
                fig_file = fig_dir / fname
                im_file = results_dir / fname
                df.at[i, f"{t_col}_tex"] = f"\\includegraphics[width=3cm]{{{fig_file}}}"
                print(f"Saving image for row {i} to {im_file} and referencing as {fig_file} in LaTeX", r[t_col])
                img: Image = r[t_col]
                img.save(im_file)
            else:
                df.at[i, f"{t_col}_tex"] = "No image"
    df_renamed = df.rename(columns=col_mapping)
    print(f"Renamed columns for LaTeX: {df_renamed.columns}")
    allowed_cols = [c for c in list(col_mapping.values()) if c in df_renamed.columns]
    print(f"Allowed columns for LaTeX output: {allowed_cols}")
    df_renamed = df_renamed[allowed_cols]
    df_renamed.set_index(allowed_cols[0], inplace=True)
    df_renamed.style.to_latex(
        buf= results_dir / out_name,
        column_format=col_format,
    )


to_tex_with_thumbs(easy_results)

Saving image for row 0 to ../scratch/results/dbpedia/media/thumb_img_0.png and referencing as figures/generated/dbpedia/media/thumb_img_0.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x247 at 0x7FF50012B5C0>
Saving image for row 1 to ../scratch/results/dbpedia/media/thumb_img_1.png and referencing as figures/generated/dbpedia/media/thumb_img_1.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x217 at 0x7FF500128050>
Saving image for row 2 to ../scratch/results/dbpedia/media/thumb_img_2.png and referencing as figures/generated/dbpedia/media/thumb_img_2.png in LaTeX <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=330x250 at 0x7FF4FBFE3360>
Renamed columns for LaTeX: Index(['Ship', 'thumb_emb', 'Distance', 'thumb', 'thumb_img', 'Thumbnail'], dtype='str')
Allowed columns for LaTeX output: ['Ship', 'Thumbnail', 'Distance']
